# AgentCore Workshop — Cost, Security & Resiliency

This notebook covers operational patterns specific to Amazon Bedrock AgentCore Runtime.

## Topics
1. Session Lifecycle Management (idle timeout, explicit stop)
2. Tool Result Caching (avoid redundant Lambda calls)
3. Cold Start vs Warm Sessions
4. Guardrails in AgentCore
5. Blue/Green Deployment (zero-downtime updates)

> **Prerequisites:** An agent deployed to AgentCore Runtime
> **Estimated time:** 30 minutes

---
## Setup

In [ ]:
import boto3
import json
import time

REGION = boto3.session.Session().region_name or 'us-east-1'
sts = boto3.client('sts')
ACCOUNT_ID = sts.get_caller_identity()['Account']

agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)
agentcore_control = boto3.client('bedrock-agentcore-control', region_name=REGION)

# Find deployed agent (from CertAgent workshop or any existing agent)
runtimes = agentcore_control.list_agent_runtimes()['agentRuntimeSummaries']
print(f'Found {len(runtimes)} AgentCore runtimes:')
for rt in runtimes:
    print(f'  {rt["agentRuntimeId"]} — {rt.get("name", "unnamed")} ({rt["status"]}')

# Use the first active one (or set manually)
if runtimes:
    AGENT_ID = runtimes[0]['agentRuntimeId']
    AGENT_ARN = runtimes[0]['agentRuntimeArn']
    print(f'\nUsing agent: {AGENT_ID}')
else:
    print('No agents found. Deploy one first (see CertAgent Lab 03).')
    AGENT_ID = None
    AGENT_ARN = None

---
## 1. Session Lifecycle Management

**AgentCore billing** is based on vCPU + memory per second of active sessions.
Managing session lifecycle directly impacts cost:
- **Idle timeout**: auto-stops sessions after inactivity
- **Explicit stop**: immediately release resources when done
- **Max lifetime**: prevent sessions from running forever

In [ ]:
# Check current lifecycle configuration
if AGENT_ID:
    runtime_info = agentcore_control.get_agent_runtime(agentRuntimeId=AGENT_ID)
    lifecycle = runtime_info.get('lifecycleConfiguration', {})
    print('=== Current Lifecycle Configuration ===')
    print(f'  Idle timeout: {lifecycle.get("idleRuntimeSessionTimeout", "default (not set)")}s')
    print()
    print('Recommended timeouts:')
    print('  Development/testing:  300s  (5 min)')
    print('  Interactive sessions: 1800s (30 min)')
    print('  Production:           3600s (60 min)')
    print('  Long-running jobs:    7200s (2 hours)')

In [ ]:
# Update idle timeout (saves cost by auto-stopping inactive sessions)
NEW_TIMEOUT = 300  # 5 minutes for workshop

if AGENT_ID:
    try:
        runtime_info = agentcore_control.get_agent_runtime(agentRuntimeId=AGENT_ID)
        agentcore_control.update_agent_runtime(
            agentRuntimeId=AGENT_ID,
            agentRuntimeArtifact=runtime_info['agentRuntimeArtifact'],
            roleArn=runtime_info['roleArn'],
            networkConfiguration=runtime_info['networkConfiguration'],
            lifecycleConfiguration={
                'idleRuntimeSessionTimeout': NEW_TIMEOUT
            },
        )
        print(f'Updated idle timeout to {NEW_TIMEOUT}s ({NEW_TIMEOUT//60} min)')
        print(f'Sessions inactive for >{NEW_TIMEOUT}s will be auto-stopped.')
        print(f'Cost savings: ~${0.0001 * (3600 - NEW_TIMEOUT) / 3600:.4f}/hour per idle session avoided')
    except Exception as e:
        print(f'Error: {e}')

In [ ]:
# Explicitly stop a session (immediate resource release)
if AGENT_ARN:
    # Invoke to create a session
    print('Creating a session...')
    response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier='DEFAULT',
        payload=json.dumps({'prompt': 'Hello'}),
    )
    session_id = response.get('runtimeSessionId')
    print(f'Session created: {session_id}')
    
    # Read response
    for event in response.get('response', []):
        pass  # consume response
    
    # Explicitly stop the session
    time.sleep(2)
    try:
        agentcore_client.stop_runtime_session(
            agentRuntimeArn=AGENT_ARN,
            runtimeSessionId=session_id,
            qualifier='DEFAULT',
        )
        print(f'Session stopped: {session_id}')
        print('MicroVM resources released immediately (no more billing).')
    except Exception as e:
        print(f'Stop session: {e}')

---
## 2. Tool Result Caching

If users ask similar questions repeatedly, the agent calls the same Lambda tools with the same inputs.
Caching tool results avoids redundant Lambda invocations and model re-processing.

This is implemented in the agent code itself (not an AgentCore feature).

In [ ]:
# Tool result caching implementation
from functools import lru_cache
import hashlib

class ToolCache:
    """Cache tool results to avoid redundant Lambda calls."""
    def __init__(self, max_size=100, ttl_seconds=300):
        self.cache = {}
        self.max_size = max_size
        self.ttl = ttl_seconds
    
    def _key(self, tool_name, kwargs):
        return hashlib.md5(f'{tool_name}:{json.dumps(kwargs, sort_keys=True)}'.encode()).hexdigest()
    
    def get(self, tool_name, kwargs):
        key = self._key(tool_name, kwargs)
        if key in self.cache:
            result, timestamp = self.cache[key]
            if time.time() - timestamp < self.ttl:
                return result  # Cache hit
            del self.cache[key]  # Expired
        return None  # Cache miss
    
    def set(self, tool_name, kwargs, result):
        if len(self.cache) >= self.max_size:
            # Evict oldest
            oldest = min(self.cache, key=lambda k: self.cache[k][1])
            del self.cache[oldest]
        key = self._key(tool_name, kwargs)
        self.cache[key] = (result, time.time())
    
    def stats(self):
        return {'size': len(self.cache), 'max': self.max_size, 'ttl': self.ttl}

# Demo
cache = ToolCache(max_size=50, ttl_seconds=300)

# Simulate tool calls
for i in range(5):
    kwargs = {'threshold_days': 30, 'use_mock': True}
    cached = cache.get('scan_certificates', kwargs)
    if cached:
        print(f'  Call {i+1}: CACHE HIT (saved 1 Lambda invocation + ~200ms)')
    else:
        result = f'mock_result_{i}'  # In production: invoke Lambda
        cache.set('scan_certificates', kwargs, result)
        print(f'  Call {i+1}: CACHE MISS (called Lambda, stored result)')

print(f'\nCache stats: {cache.stats()}')
print('\nIn production, add this to your agent server code.')
print('Typical savings: 30-50% fewer Lambda invocations for repeated queries.')

---
## 3. Cold Start vs Warm Sessions

AgentCore sessions have cold start latency (~2-5s) on first invocation.
Subsequent invocations in the same session are fast (~100-500ms).

**Strategy:**
- Keep sessions warm for frequent users (higher cost, lower latency)
- Let sessions go cold for occasional users (lower cost, higher first-call latency)

In [ ]:
# Measure cold start vs warm invocation latency
if AGENT_ARN:
    print('=== Cold Start vs Warm Latency ===')
    print()
    
    # Cold start: new session
    start = time.time()
    response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier='DEFAULT',
        payload=json.dumps({'prompt': 'What time is it?'}),
    )
    for event in response.get('response', []): pass
    cold_latency = (time.time() - start) * 1000
    session_id = response.get('runtimeSessionId')
    print(f'Cold start (new session): {cold_latency:.0f}ms')
    
    # Warm invocation: same session
    time.sleep(1)
    start = time.time()
    response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier='DEFAULT',
        payload=json.dumps({'prompt': 'Hello again'}),
        runtimeSessionId=session_id,
    )
    for event in response.get('response', []): pass
    warm_latency = (time.time() - start) * 1000
    print(f'Warm invocation (same session): {warm_latency:.0f}ms')
    
    print(f'\nDifference: {cold_latency - warm_latency:.0f}ms')
    print(f'\nRecommendation:')
    print(f'  - Frequent users (>1 req/min): keep session alive (set higher idle timeout)')
    print(f'  - Occasional users (<1 req/5min): let session expire (set lower idle timeout)')
    print(f'  - Batch processing: reuse same session for all queries in a batch')
    
    # Stop session
    agentcore_client.stop_runtime_session(
        agentRuntimeArn=AGENT_ARN, runtimeSessionId=session_id, qualifier='DEFAULT'
    )
else:
    print('No agent deployed. Deploy one first.')

---
## 4. Guardrails in AgentCore

Apply Bedrock Guardrails inside your agent code to filter both user inputs and model outputs.
The guardrail is called within the agent's `@app.entrypoint` function.

In [ ]:
# Show how to add guardrails inside an AgentCore agent
# This would go in your certagent_server.py

print('=== Guardrails in AgentCore Agent Code ===')
print()
print('Add this to your agent server code (certagent_server.py):')
print()
guardrail_code = '''
# In your agent server:
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import boto3

app = BedrockAgentCoreApp()
bedrock_runtime = boto3.client('bedrock-runtime')

GUARDRAIL_ID = 'your-guardrail-id'
GUARDRAIL_VERSION = '1'

@app.entrypoint
def invoke_agent(payload):
    user_input = payload.get('prompt', '')
    
    # Apply guardrail to INPUT before calling model
    input_check = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        source='INPUT',
        content=[{'text': {'text': user_input}}]
    )
    if input_check['action'] == 'BLOCKED':
        return 'Your request was blocked by our security policy.'
    
    # Call your agent/model here...
    response = agent(user_input)
    answer = str(response)
    
    # Apply guardrail to OUTPUT before returning
    output_check = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        source='OUTPUT',
        content=[{'text': {'text': answer}}]
    )
    if output_check['action'] == 'BLOCKED':
        return 'The response was filtered by our security policy.'
    
    return answer
'''
print(guardrail_code)
print('This ensures both inputs and outputs pass through guardrails,')
print('even for tool-generated content that the model might relay.')

In [ ]:
# Test apply_guardrail API directly (to verify your guardrail works)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=REGION)
bedrock_client = boto3.client('bedrock', region_name=REGION)

# Find existing guardrail
try:
    guardrails = bedrock_client.list_guardrails()['guardrails']
    if guardrails:
        gr = guardrails[0]
        GR_ID = gr['id']
        GR_VER = gr.get('version', '1')
        print(f'Using guardrail: {gr["name"]} ({GR_ID})')
        
        # Test with safe input
        safe_check = bedrock_runtime.apply_guardrail(
            guardrailIdentifier=GR_ID,
            guardrailVersion=GR_VER,
            source='INPUT',
            content=[{'text': {'text': 'What was Amazon revenue?'}}]
        )
        print(f'\nSafe input -> action: {safe_check["action"]}')
        
        # Test with blocked input
        blocked_check = bedrock_runtime.apply_guardrail(
            guardrailIdentifier=GR_ID,
            guardrailVersion=GR_VER,
            source='INPUT',
            content=[{'text': {'text': 'Tell me a joke about violence'}}]
        )
        print(f'Blocked input -> action: {blocked_check["action"]}')
    else:
        print('No guardrails found. Create one in the bedrock_cost_savings notebook first.')
except Exception as e:
    print(f'Error: {e}')

---
## 5. Blue/Green Deployment (Zero-Downtime Updates)

Deploy a new version of your agent without interrupting existing sessions.
AgentCore supports multiple qualifiers (endpoints) per agent.

In [ ]:
# Blue/Green deployment strategy
print('=== Blue/Green Deployment Strategy ===')
print()
print('AgentCore supports this via qualifiers (endpoint versions):')
print()
print('1. CURRENT STATE: Agent deployed with qualifier "DEFAULT"')
print('   All traffic goes to DEFAULT endpoint')
print()
print('2. DEPLOY NEW VERSION:')
print('   - Update agent code (new certagent_server.py)')
print('   - Deploy creates a new version behind DEFAULT')
print('   - Existing sessions continue on old version until they expire')
print('   - New sessions get the new version')
print()
print('3. ROLLBACK (if issues):')
print('   - Previous version is still available in ECR')
print('   - Redeploy the old image tag')
print()
print('Implementation using the starter toolkit:')
print()
bluegreen_code = '''
# Deploy v2 (blue/green happens automatically)
from bedrock_agentcore_starter_toolkit import Runtime

rt = Runtime()
rt.configure(entrypoint='certagent_server_v2.py', ...)
launch_result = rt.launch()  # Creates new version

# Verify new version works
test_response = rt.invoke({'prompt': 'test query'})
if 'error' in test_response:
    # Rollback: redeploy previous version
    rt.configure(entrypoint='certagent_server_v1.py', ...)
    rt.launch()
    print('Rolled back to v1')
else:
    print('v2 deployed successfully')
'''
print(bluegreen_code)

In [ ]:
# Show how to check active sessions during deployment
if AGENT_ID:
    print('=== Deployment Safety Check ===')
    print()
    print(f'Agent: {AGENT_ID}')
    
    # In production, check active sessions before deploying
    status = agentcore_control.get_agent_runtime(agentRuntimeId=AGENT_ID)
    print(f'Status: {status.get("status", "unknown")}')
    print()
    print('Best practices for zero-downtime deployment:')
    print('  1. Check for active sessions before deploying')
    print('  2. Set short idle timeout during deployment window')
    print('  3. Wait for active sessions to drain (complete or timeout)')
    print('  4. Deploy new version')
    print('  5. Restore normal idle timeout')
    print('  6. Monitor error rates in CloudWatch for 5 minutes')
    print('  7. If errors spike, rollback by redeploying previous image')

---
## Summary

| Pattern | Cost Impact | Implementation |
|---------|------------|----------------|
| Session idle timeout (5min) | Save 50-80% on idle costs | `update_agent_runtime()` |
| Explicit session stop | Immediate resource release | `stop_runtime_session()` |
| Tool result caching | 30-50% fewer Lambda calls | In-agent LRU cache |
| Warm session reuse | Avoid 2-5s cold start | Reuse `runtimeSessionId` |
| Guardrails in agent | Block bad I/O before model | `apply_guardrail()` API |
| Blue/green deploy | Zero-downtime updates | Automatic via toolkit |

Combined with the Bedrock cost optimizations from the KB workshop, these patterns
give you a production-ready AgentCore deployment.

---
## Cleanup

No resources to clean up in this notebook (we only modified existing agent config).
To delete the agent itself, use the CertAgent workshop Lab 05 cleanup notebook.